# 정상 / 언더필 분류 — Colab 학습 (KEEP 3,425장 · 마스킹 · 500px)

**준비:** Drive `산공종설/` 폴더에 `keep3425_masked_500.zip`을 올린다. 그다음 위에서부터 셀을 하나씩 실행한다(Shift+Enter).

zip 안에는 세 가지가 있다.
- 마스킹 크롭 3,425장: normal 1714장, underfill_50 1711장
- `folds.csv`: 시간 블록 5-fold 분할표
- `train_pilot_classifier.py`: 학습 스크립트

⚠️ 여기서 나오는 점수는 **예비 점수**다. 클래스마다 연속 촬영 한 번 안에서 시간 구간을 나눠 검증한 것이라, 새 출력 실험에서의 성능이 아니다.

## 1. GPU 확인
런타임이 GPU인지 확인한다. `assert`는 조건이 거짓이면 그 자리에서 멈추고 메시지를 띄운다. 모르고 CPU로 몇 시간 돌리는 걸 막아 준다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결
Colab은 실행할 때마다 빈 컴퓨터를 새로 빌려 준다. 그래서 Drive를 붙여야 내 파일이 보인다. 권한 창이 뜨면 허용한다.

결과는 Drive(`results/`)에 바로 저장한다. 런타임이 끊기면 `/content`에 있던 파일은 전부 사라지기 때문이다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"   # zip을 올린 Drive 폴더
ZIP_NAME = "keep3425_masked_500.zip"
RESULTS_DIR = f"{DRIVE_DIR}/results"
print(os.listdir(DRIVE_DIR))
assert os.path.exists(f"{DRIVE_DIR}/{ZIP_NAME}"), "zip이 안 보임: DRIVE_DIR 경로를 확인"

## 3. zip 복사 → 압축 해제
Drive에서 사진 3천여 개를 하나씩 읽으면 느리다. 그래서 zip 하나를 Colab 로컬 디스크(`/content`)로 복사한 뒤 푼다.

- `!`로 시작하는 줄은 파이썬이 아니라 리눅스 셸 명령이다.
- `{DATA}`처럼 중괄호로 감싸면 파이썬 변수 값이 그 자리에 들어간다.
- 마지막 줄에 3425가 나와야 정상이다.

In [ ]:
DATA = "/content/data"
!mkdir -p {DATA}
!cp "{DRIVE_DIR}/{ZIP_NAME}" /content/
!unzip -q -o /content/{ZIP_NAME} -d {DATA}
!ls {DATA}
!find {DATA}/size_500/masked_crops -name "*.jpg" | wc -l

## 4. 1 epoch 점검 + 소요 시간 추정
본 실행은 몇 시간 걸릴 수 있다. 먼저 fold 0을 1 epoch만 돌려서 두 가지를 확인한다.
1. 에러 없이 끝나는지
2. 1 epoch에 몇 초 걸리는지 → 이걸로 전체 시간을 추정한다

여기서 찍히는 정확도는 1 epoch짜리라 의미가 없다.

In [ ]:
!rm -rf /content/smoke
!python {DATA}/train_pilot_classifier.py --folds-csv {DATA}/folds.csv --pilot-root {DATA} --output /content/smoke --arch efficientnet_b0 --img-size 500 --folds 0 --seeds 0 --epochs 1

import json
sec = {m: json.load(open(f"/content/smoke/runs/{m}/seed0/fold0/metrics.json"))["seconds"] for m in ("linear", "finetune")}
# 1 epoch 시간 × 본 실행 epoch 수(linear 30, finetune 20) × 5 fold × 시드 3개
est = (sec["linear"] * 30 + sec["finetune"] * 20) * 5 * 3 / 3600
print(sec, f"-> 시드 3개 전체 약 {est:.1f}시간, 시드 1개면 약 {est / 3:.1f}시간 (EfficientNet-B0 기준)")

## 5. 본 실행
**무엇을 하나**
- **분할(folds.csv):** 클래스마다 사진을 프레임 번호순으로 줄 세워 5덩어리로 자른다. fold k는 k번째 덩어리로 검증하고 나머지로 학습한다. 검증 사진과 30프레임 이내인 학습 사진은 빼 둔다(buffer). 1초 간격 사진은 거의 똑같아서, 이웃 사진이 학습과 검증에 나뉘어 들어가면 점수가 부풀려지기 때문이다.
- **모드 2개:**
  - `linear`: ImageNet으로 미리 학습된 몸통은 고정하고, 마지막 분류층만 새로 학습한다. 빠르고 과적합이 적다.
  - `finetune`: 몸통까지 작은 학습률로 조금씩 고친다.
- **`--img-size 500`:** 500 크롭을 줄이지 않고 그대로 넣는다. 가는 틈이 뭉개지지 않는다.
- **시드 3개:** 무작위 요소(초기값, 섞는 순서)만 바꿔 3번 반복한다. 점수가 얼마나 흔들리는지 보는 용도다.

**주의**
- 출력 폴더가 이미 있으면 스크립트가 멈춘다(덮어쓰기 방지). 다시 돌리려면 `OUT` 이름을 바꾼다.
- ResNet18과 비교하려면 `ARCH = "resnet18"`로 바꾸고 이 셀부터 다시 실행한다.
- 4번 셀에서 시간이 너무 길게 나오면 `SEEDS = "0"`으로 먼저 돌린다.

In [ ]:
ARCH = "efficientnet_b0"   # 또는 "resnet18"
SEEDS = "0 1 2"            # 시간이 모자라면 "0"
OUT = f"{RESULTS_DIR}/{ARCH}_keep3425_masked_500_seeds{SEEDS.replace(' ', '')}"
print(OUT)
!python {DATA}/train_pilot_classifier.py --folds-csv {DATA}/folds.csv --pilot-root {DATA} --output "{OUT}" --arch {ARCH} --img-size 500 --seeds {SEEDS}

## 6. 결과 보기
**표 읽는 법**
- `pooled_…`: 5개 fold의 검증 예측을 모아서 한 번에 계산한 점수다. 모든 사진이 한 번씩 검증된다.
- `underfill_recall`: 실제 언더필 사진 중 모델이 잡아낸 비율이다. 불량을 놓치지 않는 정도다.
- `normal_recall`: 실제 정상 사진을 정상으로 맞힌 비율이다.
- `auroc`: 판정 기준(임계값)과 상관없이 두 클래스를 얼마나 잘 가르는지 나타낸다. 0.5는 찍기 수준, 1은 완벽이다.
- `train_accuracy`는 1.0 가까운데 검증 점수가 훨씬 낮으면 과적합(학습 사진을 외운 것)이다.

**shortcut_check**
마스킹 비율이나 평균 밝기 하나만으로 클래스가 갈리는지 본다.
- AUROC가 0.5 근처면 괜찮다.
- 0이나 1에 가까우면, 모델이 결함 대신 그 차이를 보고 맞힐 위험이 있다.

**결과 공유:** 이 셀 출력을 그대로 복사해서 Claude에게 붙여 주면 된다.

In [ ]:
import pandas as pd
cols = ["mode", "runs", "pooled_mean_accuracy", "pooled_mean_underfill_recall", "pooled_mean_underfill_f1",
        "pooled_mean_normal_recall", "pooled_mean_auroc", "fold_std_accuracy", "mean_train_accuracy",
        "mean_seconds_per_run", "max_peak_alloc_mib"]
print(pd.read_csv(f"{OUT}/summary.csv", encoding="utf-8-sig")[cols].round(3).to_string(index=False))
fm = pd.read_csv(f"{OUT}/fold_metrics.csv", encoding="utf-8-sig")
print(fm[["mode", "seed", "fold", "val_accuracy", "val_underfill_recall", "val_normal_recall", "val_auroc", "train_accuracy"]].round(3).to_string(index=False))
print(open(f"{OUT}/shortcut_check.json").read())